# satreg: fine-tuning on a Kaggle GPU

Runs `satreg train` on this notebook's GPU, once per entry of `RUNS`, and saves
each run's directory (`checkpoint.pt`, `best.pt`, `log.csv`, `validation.csv`,
`summary.md`, `context.json`) as the notebook's output, under
`outputs/train/<run>/`. The procedure, from the timing run to resuming a run in
another session, is in `docs/kaggle.md` in the repository.

Settings: accelerator **GPU T4 x2** or **GPU T4** (a training uses one GPU),
internet **on**. Inputs: the training chips dataset (`satreg-training-chips`),
and, to go on with a run, the output of the version of this notebook that
started it. While the repository is private, attach a Kaggle secret named
`GITHUB_TOKEN` holding a read-only GitHub token for it.

In [ ]:
# What to run. The commit must be pushed to GitHub; context.json records the
# one that ran. Its configs/train-sites.toml, configs/sites.toml and
# configs/real.toml must be the ones the chips dataset was built with.
REPOSITORY = "https://github.com/esamoun/ai-satellite-image-registration.git"
COMMIT = "main"
# The run config: its [train] table sets the run, its [pairs] table the pairs.
CONFIG = "configs/train-timing.toml"
# One training per entry, in this order: the name of its directory under
# outputs/train/, and the options of `satreg train` that override the config.
# A run whose directory is found in an input, with a checkpoint, is resumed.
RUNS = {
    "timing-large": ["--model", "large"],
    "timing-small": ["--model", "small"],
    # The resume test: stopped half-way here, resumed by a second session
    # without --stop-at, then compared with timing-large.
    "timing-large-cut": ["--model", "large", "--stop-at", "150"],
}
# Minutes the trainings may take in all, short of the session's limit: a run
# still going then stops on a checkpoint, and the next session resumes it.
MAX_MINUTES = 600
# Pairs of runs whose loss curves are compared at the end, each read from this
# session or from an input, e.g. [("timing-large", "timing-large-cut")].
COMPARE = []

In [ ]:
import csv
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

# Kaggle saves /kaggle/working as the notebook's output; the code and the chips
# stay outside it.
WORKING = Path("/kaggle/working")
CODE = Path("/tmp/satreg")


def run(*command, cwd=None, env=None):
    """Run a command, print its output, stop the notebook if it fails.

    The output is read and printed line by line: a child process writes to the
    kernel's own streams, which Kaggle shows in the Logs tab, not in the cell.
    """
    print("$", " ".join(map(str, command)), flush=True)
    with subprocess.Popen(
        list(map(str, command)),
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command[:3])


def timed(step, started):
    print(f"{step}: {time.perf_counter() - started:.0f} s", flush=True)


SESSION_STARTED = time.perf_counter()
INPUT = Path("/kaggle/input")
TRAIN = WORKING / "outputs" / "train"
# A writable copy of the chips: the inputs are read-only.
DATA = Path("/tmp/satreg-data")

In [ ]:
run("nvidia-smi")
print("CPU cores:", os.cpu_count())

In [ ]:
# Clone, with the token of the GITHUB_TOKEN secret if one is attached. The token
# goes to git through the environment only: it is in no command line, in no
# file of the clone, and in no output.
started = time.perf_counter()
try:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:  # no secret attached, or not on Kaggle: a public repository
    token = None

git, env = ["git"], dict(os.environ)
if token:
    env["GITHUB_TOKEN"] = token
    helper = '!f() { echo username=x-access-token; echo "password=$GITHUB_TOKEN"; }; f'
    # An empty helper first drops any helper the machine configured, which
    # git would otherwise ask before this one.
    git += ["-c", "credential.helper=", "-c", f"credential.helper={helper}"]
shutil.rmtree(CODE, ignore_errors=True)
run(*git, "clone", "--quiet", REPOSITORY, CODE, env=env)
del token, env
run("git", "checkout", "--quiet", COMMIT, cwd=CODE)
run("git", "log", "-1", "--format=%H %s", cwd=CODE)
timed("clone", started)

In [ ]:
# The dl extra, then the pretrained weights a training starts from, downloaded
# once per session to PyTorch's cache, ~/.cache/torch, outside the saved output.
started = time.perf_counter()
run("make", "install-dl", cwd=CODE)
timed("install and weights", started)

In [ ]:
# The training chips: the folder of the dataset that holds chips.json and
# real/, wherever Kaggle mounted it. Given as --data-dir, every date is in the
# cache and nothing is fetched.
datasets = [p.parent for p in INPUT.rglob("chips.json") if (p.parent / "real").is_dir()]
if len(datasets) != 1:
    raise SystemExit(f"need the training chips dataset as an input, found {datasets}")
print("training chips:", datasets[0])
shutil.rmtree(DATA, ignore_errors=True)
shutil.copytree(datasets[0], DATA, copy_function=shutil.copyfile)
chips = sorted(p.relative_to(DATA) for p in DATA.rglob("*.tif"))
print(len(chips), "files, e.g.", chips[0])

In [ ]:
# Runs to go on with: the directory of a run of RUNS found in an input (the
# output of an earlier version of this notebook) is copied where this session
# writes, and the run is resumed from its checkpoint.
for name in RUNS:
    found = sorted(INPUT.rglob(f"outputs/train/{name}/checkpoint.pt"))
    if len(found) > 1:
        raise SystemExit(f"{name}: several checkpoints in the inputs: {found}")
    if found and not (TRAIN / name / "checkpoint.pt").exists():
        (TRAIN / name).mkdir(parents=True, exist_ok=True)
        for path in found[0].parent.iterdir():
            shutil.copyfile(path, TRAIN / name / path.name)
        print(f"{name}: to resume, from {found[0].parent}")

In [ ]:
for name, options in RUNS.items():
    left = MAX_MINUTES - (time.perf_counter() - SESSION_STARTED) / 60
    if left < 1:
        print(f"{name}: not started, the session's {MAX_MINUTES} min are used up")
        continue
    started = time.perf_counter()
    out = TRAIN / name
    command = ["train", "--config", CONFIG, "--device", "cuda", "--data-dir", DATA]
    command += ["--out", out, "--max-minutes", f"{left:.1f}"]
    if (out / "checkpoint.pt").exists():
        command.append("--resume")
    run(sys.executable, "-m", "satreg", *command, *options, cwd=CODE)
    timed(name, started)

In [ ]:
# What each session of each run did, and whether the chips all came from the
# dataset: a file added to the data directory was fetched from the catalogue.
for name in RUNS:
    path = TRAIN / name / "context.json"
    if not path.exists():
        continue
    context = json.loads(path.read_text())
    session = context["sessions"][-1]
    kept = (context.get("validation") or {}).get("kept") or {}
    print(name, json.dumps({
        "commit": (session.get("code") or {}).get("commit"),
        "gpu": (session.get("gpu") or {}).get("name"),
        "steps": session["steps"],
        "timing": session["timing"],
        "peak_gpu_mib": session["peak_gpu_mib"],
        "kept": {key: kept.get(key) for key in ("step", "score")},
    }, indent=2))
fetched = sorted(set(p.relative_to(DATA) for p in DATA.rglob("*.tif")) - set(chips))
print("fetched besides the dataset:", fetched or "nothing")
print(f"session: {(time.perf_counter() - SESSION_STARTED) / 60:.1f} min")

In [ ]:
# Loss curves side by side, by stretches of steps: a run that was cut and
# resumed against one that was not. On a GPU the two differ in the last digits
# from the first step on, and stay close; they are equal only on a CPU.
def losses(name):
    paths = [TRAIN / name / "log.csv", *INPUT.rglob(f"outputs/train/{name}/log.csv")]
    with next(p for p in paths if p.exists()).open(newline="") as f:
        return {int(row["step"]): float(row["loss"]) for row in csv.DictReader(f)}


for one, other in COMPARE:
    a, b = losses(one), losses(other)
    steps = sorted(set(a) & set(b))
    print(f"steps | {one} | {other} | largest gap   ({len(steps)} steps in both)")
    for i in range(0, len(steps), 25):
        part = steps[i : i + 25]
        mean_a = sum(a[s] for s in part) / len(part)
        mean_b = sum(b[s] for s in part) / len(part)
        gap = max(abs(a[s] - b[s]) for s in part)
        print(f"{part[0]}-{part[-1]} | {mean_a:.4f} | {mean_b:.4f} | {gap:.4f}")